In [1]:
import pandas as pd, numpy as np
import vivarium_inputs
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "India"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"

In [4]:
location = location.title()

In [5]:
pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2021        2022        1.975581e+05
                  0.019178   0.076712    2021        2022        5.872100e+05
                  0.076712   0.500000    2021        2022        4.326738e+06
                  0.500000   1.000000    2021        2022        5.085507e+06
                  1.000000   2.000000    2021        2022        1.035319e+07
                  2.000000   5.000000    2021        2022        3.244181e+07
                  5.000000   10.000000   2021        2022        5.860247e+07
                  10.000000  15.000000   2021        2022        6.308955e+07
                  15.000000  20.000000   2021        2022        6.421417e+07
                  20.000000  25.000000   2021        2022        6.367111e+07
                  25.000000  30.000000   2021        2022        5.987495e+07
                  30.000000  35.000000   2021        2022        5.583203e+07
  

In [6]:
asfr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location
).value
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end  parameter  
India     Female  10.0       15.0     2021        2022      lower_value    0.000183
                                                            mean_value     0.000377
                                                            upper_value    0.000709
                  15.0       20.0     2021        2022      lower_value    0.008209
                                                            mean_value     0.009704
                                                            upper_value    0.011424
                  20.0       25.0     2021        2022      lower_value    0.092077
                                                            mean_value     0.106397
                                                            upper_value    0.122441
                  25.0       30.0     2021        2022      lower_value    0.121804
                                                            mean_value     0.132412
    

In [7]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2021        2022        0.000000
                  0.019178   0.076712    2021        2022        0.000000
                  0.076712   0.500000    2021        2022        0.000000
                  0.500000   1.000000    2021        2022        0.000000
                  1.000000   2.000000    2021        2022        0.000000
                  2.000000   5.000000    2021        2022        0.000000
                  5.000000   10.000000   2021        2022        0.000000
                  10.000000  15.000000   2021        2022        0.000377
                  15.000000  20.000000   2021        2022        0.009704
                  20.000000  25.000000   2021        2022        0.106397
                  25.000000  30.000000   2021        2022        0.132412
                  30.000000  35.000000   2021        2022        0.083041
                  35.000000  40.000000   2021     

In [8]:
births = pop * asfr
births[births > 0]

location  sex     age_start  age_end  year_start  year_end
India     Female  10.0       15.0     2021        2022        2.375772e+04
                  15.0       20.0     2021        2022        6.231167e+05
                  20.0       25.0     2021        2022        6.774403e+06
                  25.0       30.0     2021        2022        7.928164e+06
                  30.0       35.0     2021        2022        4.636349e+06
                  35.0       40.0     2021        2022        1.771345e+06
                  40.0       45.0     2021        2022        4.989182e+05
                  45.0       50.0     2021        2022        1.269695e+05
                  50.0       55.0     2021        2022        1.021220e+04
Name: value, dtype: float64

In [9]:
births = births.sum()
f"{int(births):,}"

'22,393,235'

In [10]:
sim_baseline_births = pd.read_parquet(
    f"../../0200_pregnancy_sim/sim_results/{vehicle}/{location.lower()}/pregnancy_outcome_count.parquet"
)
sim_baseline_births

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,1,intervention,191,0,0.0
1,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,2,intervention,191,0,0.0
2,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,3,intervention,191,0,1.0
3,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,4,intervention,191,0,0.0
4,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,5,intervention,191,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
809995,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,1,baseline,45,0,0.0
809996,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,2,baseline,45,0,0.0
809997,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,3,baseline,45,0,0.0
809998,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,4,baseline,45,0,0.0


In [11]:
sim_baseline_births = sim_baseline_births[
    (sim_baseline_births.scenario == "baseline")
    & (sim_baseline_births.sub_entity == "live_birth")
]
sim_baseline_births

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
2705,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,1,baseline,168,0,9.0
2706,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,2,baseline,168,0,6.0
2707,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,3,baseline,168,0,6.0
2708,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,4,baseline,168,0,5.0
2709,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,5,baseline,168,0,4.0
...,...,...,...,...,...,...,...,...,...,...,...
809990,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,1,baseline,45,0,0.0
809991,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,2,baseline,45,0,0.0
809992,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,3,baseline,45,0,0.0
809993,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,4,baseline,45,0,0.0


In [12]:
sim_baseline_births = sim_baseline_births.groupby("input_draw").value.sum().mean()
sim_baseline_births

7221355.0

In [13]:
births

22393235.48415845

In [14]:
scalar = births / sim_baseline_births
scalar

3.100974191707574

In [15]:
for result in ["ylds", "ylls", "deaths", "person_time"]:
    df = pd.read_parquet(
        f"../../0300_child_sim/sim_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    df.value *= scalar
    path = pathlib.Path(
        f"../results/rescaled_child_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)